# Decision Mining

Decision mining, approach based on Leoni & van der Aalst (2013) and Rozinat & van der Aalst (2006)

Idea:
- Collect training data for every decision point in the process model (what was the state of the case before the decision and which branch was taken)
- Learn a decision tree for every decision point and turn it into rules
- Annotate the process model with the rules and probabilities of the branches

## 1. Log and model

In [ ]:
import hashlib
import pickle
from pathlib import Path

import pandas as pd
import pm4py
from pm4py.algo.conformance.alignments.petri_net import algorithm as alignments_algorithm
from pm4py.algo.conformance.alignments.petri_net.variants import state_equation_a_star as a_star
from sklearn.metrics import accuracy_score, f1_score
from sklearn.tree import DecisionTreeClassifier

from preprocessing import load_raw_log

MODEL_PATH = Path('../output/final_model.bpmn')

raw = load_raw_log()
# Same preprocessing as for the model: A_ and O_ events, W_ events reduced to their start
df_start = raw[(raw['EventOrigin'] != 'Workflow') | (raw['lifecycle:transition'] == 'start')].copy()
# Month in which the case started (1 = January 2016)
case_start = df_start.groupby('case:concept:name')['time:timestamp'].transform('min')
df_start['case:start_month'] = (case_start.dt.year - 2016) * 12 + case_start.dt.month

# pm4py gives some silent transitions random names on every conversion, so the net is cached together with its alignments
CACHE = Path(f'../output/decision_mining_{hashlib.md5(MODEL_PATH.read_bytes()).hexdigest()[:10]}.pkl')
if CACHE.exists():
    net, im, fm, alignments = pickle.loads(CACHE.read_bytes())
else:
    net, im, fm = pm4py.convert_to_petri_net(pm4py.read_bpmn(str(MODEL_PATH)))
    alignments = None

print(f"Log: {len(df_start):,} events, {df_start['case:concept:name'].nunique():,} cases")
print(f"Model: {len(net.places)} places, {len(net.transitions)} transitions ({sum(t.label is None for t in net.transitions)} silent)")

In [ ]:
CASE_VARIABLES = ['LoanGoal', 'ApplicationType', 'RequestedAmount', 'start_month']
EVENT_VARIABLES = ['OfferedAmount', 'MonthlyCost', 'NumberOfTerms', 'FirstWithdrawalAmount', 'org:resource']
TIME_VARIABLES = ['waiting_days', 'elapsed_days']
FEATURES = CASE_VARIABLES + EVENT_VARIABLES + TIME_VARIABLES + ['visit']
CATEGORICAL = ['LoanGoal', 'ApplicationType', 'org:resource']
# Whole numbers, so that thresholds like 29.5 can be shown as >= 30
INTEGER = ['start_month', 'waiting_days', 'elapsed_days', 'visit']

columns = ['case:concept:name', 'concept:name', 'time:timestamp'] + [f'case:{v}' for v in CASE_VARIABLES] + EVENT_VARIABLES
log = pm4py.convert_to_event_log(df_start[columns])
print(f'Event log with {len(log):,} traces')

## 2. Decision points

A decision point is a place with more than one outgoing arc (Rozinat et al.)

In [ ]:
def next_visible(transition, seen=None):
    seen = set() if seen is None else seen
    if transition.label:
        return {transition.label}
    if transition in seen:
        return set()
    seen.add(transition)
    found = set()
    for arc in transition.out_arcs:
        if not arc.target.out_arcs:
            found.add('end')
        for next_arc in arc.target.out_arcs:
            found |= next_visible(next_arc.target, seen)
    return found


def previous_visible(place, seen=None):
    seen = set() if seen is None else seen
    found = set()
    for arc in place.in_arcs:
        if arc.source.label:
            found.add(arc.source.label)
        elif arc.source not in seen:
            seen.add(arc.source)
            for previous_arc in arc.source.in_arcs:
                found |= previous_visible(previous_arc.source, seen)
    return found


branch_names = {t.name: t.label or 'skip to ' + ' / '.join(sorted(next_visible(t))) for t in net.transitions}
places = {p.name: p for p in net.places if len(p.out_arcs) > 1}
overview = pd.DataFrame([
    {'place': name, 'before': ', '.join(sorted(previous_visible(p))), 'branches': [branch_names[a.target.name] for a in p.out_arcs]}
    for name, p in places.items()
]).set_index('place')
overview

In [ ]:
# (activities before the decision point, one of its branches)
DECISIONS = {
    'D1': ('A_Create Application', 'A_Submitted'),
    'D2': ('A_Submitted', 'W_Handle leads'),
    'D3': ('A_Create Application, A_Submitted, W_Handle leads', 'W_Complete application'),
    'D4': ('O_Created', 'skip to O_Create Offer'),
    'D5': ('O_Sent (mail and online)', 'skip to O_Sent (mail and online)'),
    'D6': ('O_Sent (mail and online)', 'skip to O_Create Offer'),
    'D7': ('A_Complete, O_Sent (mail and online), W_Call after offers', 'W_Validate application'),
    'D8': ('A_Validating', 'W_Call incomplete files'),
    'D9': ('A_Incomplete, A_Validating', 'skip to W_Validate application'),
    'D10': ('A_Complete, A_Incomplete, A_Validating, O_Returned, O_Sent (mail and online), W_Call after offers', 'A_Denied'),
    'D11': ('A_Cancelled, A_Pending', 'skip to O_Cancelled'),
    'D12': ('O_Cancelled', 'skip to O_Cancelled'),
}


def find_place(before, branch):
    match = overview[(overview['before'] == before) & overview['branches'].map(lambda bs: any(branch in b for b in bs))]
    assert len(match) == 1, match
    return match.index[0]


place_of = {name: find_place(*key) for name, key in DECISIONS.items()}
assert sorted(place_of.values()) == sorted(places), 'every decision point needs exactly one entry'
for name, place in place_of.items():
    print(f"{name}: {overview.loc[place, 'branches']}")

## 3. Alignments

Optimal alignments with the standard cost function. The result keeps the names of silent transitions, so that skips and loop decisions can be classified (de Leoni et al.). Computing them takes a few minutes, afterwards they are read from the cache.

In [ ]:
if alignments is None:
    alignments = alignments_algorithm.apply(log, net, im, fm, parameters={a_star.Parameters.PARAM_ALIGNMENT_RESULT_IS_SYNC_PROD_AWARE: True})
    CACHE.write_bytes(pickle.dumps((net, im, fm, alignments)))

fitness = pd.Series([a['fitness'] for a in alignments])
print(f'Mean alignment fitness: {fitness.mean():.3f}')
print(f'Perfectly fitting cases: {(fitness == 1).mean():.1%}')

## 4. Training examples (de Leoni et al., Algorithm 1)

Replaying the alignment of every case:
- Every transition that leaves a decision point, as a move in model or a synchronous move, becomes one example: the values known so far and the chosen branch.
- Values are only updated by synchronous moves and only after the decision was recorded, so an example never contains information from its own decision. Moves in log are deviations and are ignored.
- Values that were not written yet are missing (e.g. offer attributes before the first offer).

Three features describe the decision itself (performance features in Park et al.):
- waiting_days: days from the last event until the decision. A decision is visible when the next event happens, so the moment of the decision is the time of that event.
- elapsed_days: days from the start of the case until the decision.
- visit: how often the case has reached this decision point, so that the trees can tell loop iterations apart (loop semantics in Rozinat et al.).

Times are counted in whole days: automated steps are logged milliseconds apart, so finer times would let the trees learn the logging order instead of a business rule. Decisions without a following event have no observable moment, so their time features are missing:
- A decision to end the case: the time features are not used for decision points with a branch to the end.
- Decisions the alignment adds after the last event, e.g. for cases that were still open when the log ended. Rules on time therefore say (or missing) where the tree sends these examples.

In [ ]:
examples = {name: [] for name in place_of}
decision_of = {t.name: name for name, place in place_of.items() for t in (a.target for a in places[place].out_arcs)}

for trace, alignment in zip(log, alignments):
    values = {v: trace.attributes.get(v) for v in CASE_VARIABLES} | dict.fromkeys(EVENT_VARIABLES, float('nan'))
    case_start = last_event_time = trace[0]['time:timestamp']
    visits = dict.fromkeys(place_of, 0)
    pending = []  # decisions whose moment is the next event
    events = iter(trace)
    for (log_name, model_name), _ in alignment['alignment']:
        if model_name in decision_of:
            name = decision_of[model_name]
            visits[name] += 1
            example = {**values, 'visit': visits[name], 'case': trace.attributes['concept:name'], 'decision': branch_names[model_name]}
            examples[name].append(example)
            pending.append(example)
        if log_name != '>>':
            event = next(events)
            for example in pending:
                example['waiting_days'] = (event['time:timestamp'] - last_event_time).days
                example['elapsed_days'] = (event['time:timestamp'] - case_start).days
            pending = []
            last_event_time = event['time:timestamp']
            if model_name != '>>':
                values.update({v: event[v] for v in EVENT_VARIABLES if pd.notna(event.get(v))})

tables = {name: pd.DataFrame(rows) for name, rows in examples.items()}
for name, table in tables.items():
    print(f"{name}: {len(table):,} examples, {table['decision'].value_counts(normalize=True).round(3).to_dict()}")

## 5. Decision trees

- One decision tree per decision point
- Depth at most 3 and at least 1% of the examples per leaf, to keep the rules readable and avoid overfitting (Rozinat et al.; Park et al. prune their trees for the same reason)
- Use 50% of cases for training and 50% for testing (de Leoni et al.)
- a rule is only established if the difference to the baseline is a least 5 percent

In [ ]:
MIN_GAIN = 0.05


def features_for(name):
    ends_case = any('end' in branch.removeprefix('skip to ').split(' / ') for branch in overview.loc[place_of[name], 'branches'])
    return [f for f in FEATURES if not (ends_case and f in TIME_VARIABLES)]


def mine_decision(name, table):
    X = pd.get_dummies(table[features_for(name)], columns=CATEGORICAL, dtype=float)
    y = table['decision']
    train = table['case'].isin(pd.Series(table['case'].unique()).sample(frac=0.5, random_state=42))
    tree = DecisionTreeClassifier(criterion='entropy', max_depth=3, min_samples_leaf=max(50, int(0.01 * train.sum())), random_state=42)
    tree.fit(X[train], y[train])
    predicted = tree.predict(X[~train])
    baseline = y[~train].value_counts(normalize=True).max()
    accuracy = accuracy_score(y[~train], predicted)
    metrics = {
        'examples': len(table),
        'baseline': round(baseline, 3),
        'accuracy': round(accuracy, 3),
        'macro F1': round(f1_score(y[~train], predicted, average='macro'), 3),
        'guard': accuracy - baseline >= MIN_GAIN,
    }
    return tree, list(X.columns), set(X.columns[X[train].isna().any()]), metrics


results = {name: mine_decision(name, table) for name, table in tables.items()}
pd.DataFrame({name: metrics for name, (*_, metrics) in results.items()}).T

## 6. Rules

- Conditions along a path are joined with AND, several leaves of the same branch with OR (Rozinat et al.)

In [ ]:
def number(value):
    return f'{value:,.0f}' if abs(value) >= 100 else f'{value:.4g}'


def describe(path, with_missing):
    parts = []
    for feature, (low, high) in path['bounds'].items():
        if feature in INTEGER:
            low, high = (None if low is None else int(low) + 1), (None if high is None else int(high))
            text = (f'{feature} = {low}' if low == high else f'{low} <= {feature} <= {high}' if low is not None and high is not None
                    else f'{feature} >= {low}' if low is not None else f'{feature} <= {high}')
        else:
            text = (f'{number(low)} < {feature} <= {number(high)}' if low is not None and high is not None
                    else f'{feature} > {number(low)}' if low is not None else f'{feature} <= {number(high)}')
        parts.append(text + (' (or missing)' if feature in with_missing and path['missing'].get(feature, True) else ''))
    for column, (equals, excluded) in path['categories'].items():
        parts.append(f'{column} = {equals}' if equals else f"{column} not in {{{', '.join(sorted(excluded))}}}" if len(excluded) > 1
                     else f'{column} != {next(iter(excluded))}')
    return ' AND '.join(parts) or 'always'


def tree_rules(tree, feature_names, with_missing):
    t = tree.tree_

    def predictions(node):
        if t.children_left[node] == -1:
            return {t.value[node][0].argmax()}
        return predictions(t.children_left[node]) | predictions(t.children_right[node])

    rules = []

    def walk(node, path):
        if len(predictions(node)) == 1:
            shares = t.value[node][0]
            rules.append({'decision': tree.classes_[shares.argmax()], 'rule': describe(path, with_missing),
                          'examples': t.n_node_samples[node], 'purity': round(shares.max(), 3)})
            return
        feature, threshold = feature_names[t.feature[node]], t.threshold[node]
        for child, goes_left in [(t.children_left[node], True), (t.children_right[node], False)]:
            branch = {'bounds': dict(path['bounds']), 'categories': dict(path['categories']), 'missing': dict(path['missing'])}
            column = next((c for c in CATEGORICAL if feature.startswith(c + '_')), None)
            if column:
                equals, excluded = branch['categories'].get(column, (None, set()))
                value = feature[len(column) + 1:]
                branch['categories'][column] = (equals, excluded | {value}) if goes_left else (value, set())
            else:
                low, high = branch['bounds'].get(feature, (None, None))
                branch['bounds'][feature] = (low, threshold if high is None else min(high, threshold)) if goes_left \
                    else (threshold if low is None else max(low, threshold), high)
                if goes_left != bool(t.missing_go_to_left[node]):
                    branch['missing'][feature] = False
            walk(child, branch)

    walk(0, {'bounds': {}, 'categories': {}, 'missing': {}})
    return pd.DataFrame(rules).sort_values(['decision', 'examples'], ascending=[True, False])


for name, (tree, feature_names, with_missing, metrics) in results.items():
    if metrics['guard']:
        print(f"=== {name} (accuracy {metrics['accuracy']} vs. baseline {metrics['baseline']})")
        print(tree_rules(tree, feature_names, with_missing).to_string(index=False))
        print()

## 7. Create Annotations for the BPMN

In [ ]:
rows = []
for name, table in tables.items():
    tree, feature_names, with_missing, metrics = results[name]
    conditions = {}
    if metrics['guard']:
        rules = tree_rules(tree, feature_names, with_missing)
        conditions = rules.groupby('decision')['rule'].agg(lambda r: ' OR '.join(f'({rule})' if len(r) > 1 else rule for rule in r))
    for path, share in table['decision'].value_counts(normalize=True).items():
        rows.append({
            'decision': name,
            'gateway id': place_of[name].removeprefix('exi_'),
            'path': path,
            'probability': round(share, 3),
            'condition': conditions.get(path, 'not predicted by any rule') if metrics['guard'] else 'no rule in the data',
        })

annotations = pd.DataFrame(rows)
annotations.to_csv('../output/bpmn_annotations.csv', index=False)
annotations

## 8. Create Annotated BPMN

In [ ]:
import re
from xml.sax.saxutils import escape

ANNOTATED_PATH = Path('../output/final_model_annotated.bpmn')
FLOW = re.compile(r'<bpmn:sequenceFlow id="([^"]+)" name="[^"]*" sourceRef="([^"]+)" targetRef="([^"]+)"')

xml = MODEL_PATH.read_text()
flows = {m.group(1): (m.group(2), m.group(3)) for m in FLOW.finditer(xml)}
assert len(flows) == xml.count('<bpmn:sequenceFlow '), 'every sequence flow must match the pattern'


def flow_of(gateway, transition, unresolved_flows):
    # Silent transitions are named after their sequence flow, visible ones after their task
    if transition.name.startswith('sfl_'):
        return transition.name.removeprefix('sfl_')
    matches = [f for f in unresolved_flows if flows[f][1] == transition.name]
    return matches[0] if len(matches) == 1 else None


def label(row):
    if row['condition'] == 'no rule in the data':
        return f"p = {row['probability']:.2f}"
    if row['condition'] == 'not predicted by any rule':
        return f"no rule (p = {row['probability']:.2f})"
    return f"{row['condition']} (p = {row['probability']:.2f})"


labels = {}
for name, place in place_of.items():
    gateway = place.removeprefix('exi_')
    outgoing = [f for f, (source, _) in flows.items() if source == gateway]
    rows = annotations[annotations['decision'] == name].set_index('path')
    transitions = [a.target for a in places[place].out_arcs]
    assignment = {}
    for t in transitions:
        flow = flow_of(gateway, t, [f for f in outgoing if f not in assignment.values()])
        if flow is not None:
            assignment[t.name] = flow
    # Some silent transitions get random names: they take the one remaining outgoing flow
    missing = [t for t in transitions if t.name not in assignment]
    remaining = [f for f in outgoing if f not in assignment.values()]
    assert len(missing) == len(remaining) <= 1, (name, missing, remaining)
    assignment.update({t.name: f for t, f in zip(missing, remaining)})
    assert sorted(assignment.values()) == sorted(outgoing), name
    for t in transitions:
        labels[assignment[t.name]] = label(rows.loc[branch_names[t.name]])

annotated = FLOW.sub(lambda m: f'<bpmn:sequenceFlow id="{m.group(1)}" name="{escape(labels.get(m.group(1), ""), {chr(34): "&quot;"})}" '
                               f'sourceRef="{m.group(2)}" targetRef="{m.group(3)}"', xml)
ANNOTATED_PATH.write_text(annotated)

check, *_ = pm4py.convert_to_petri_net(pm4py.read_bpmn(str(ANNOTATED_PATH)))
assert len(check.transitions) == len(net.transitions), 'the annotated model must have the same structure'
print(f'{len(labels)} paths annotated, written to {ANNOTATED_PATH}')
for name, place in place_of.items():
    gateway = place.removeprefix('exi_')
    for flow, (source, target) in flows.items():
        if source == gateway:
            print(f'{name}: {labels[flow]}')